# Neighborhood Renovation ROI Strategy

**Original Question:** Where would buying a property and renovating give me the best investment opportunity?

_Exported from PlotStudio_

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# PlotStudio stub — no-op outside the app
def add_to_workspace(df):
    pass


# Load source data
df_housing = pd.read_csv("data/df_housing.csv")

## Task 1: Establish a clean, feature-rich housing dataset with clear renovation and ROI-related variables to support neighborhood and modeling analysis.

**Acceptance Criteria:** An analysis-ready dataframe (df_housing_features) exists with cleaned key fields, explicit handling of MNAR missingness, engineered renovation recency and property profile features, and basic sanity checks confirming the grain, sample size, and distributions of renovation-related variables. At least one visualization clarifies the distribution of renovation recency over time.

### 1.1: Confirm the unit of analysis, availability of renovation-relevant columns, and absence of duplicates before engineering features.
_Output: print_

In [ ]:
# Print dataset shape and dtypes summary
print(f"df_housing shape: {df_housing.shape}")
print("\ndf_housing dtypes:")
print(df_housing.dtypes)

# Verify Id uniqueness
id_unique = df_housing["Id"].is_unique if "Id" in df_housing.columns else False
print(f"\nIs 'Id' unique? {id_unique}")
if "Id" in df_housing.columns:
    print(f"Number of unique Ids: {df_housing['Id'].nunique()}")

# List of renovation-relevant columns
renovation_cols = [
    "SalePrice",
    "Neighborhood",
    "YearBuilt",
    "YearRemodAdd",
    "OverallQual",
    "KitchenQual",
    "GrLivArea",
    "FullBath",
    "HalfBath",
    "BsmtFullBath",
    "BsmtHalfBath",
    "GarageCars",
    "YrSold",
    "MoSold",
]

# Show non-missing counts for these columns
print("\nNon-missing value counts for renovation-relevant columns:")
print(df_housing[renovation_cols].notnull().sum())

# Show a few example rows from these columns
print("\nExample rows (first 5) for renovation-relevant columns:")
print(df_housing[renovation_cols].head())


### 1.2: Create a cleaned base dataframe with non-informative high-missingness columns removed and MNAR missing values encoded meaningfully for renovation analysis.
_Output: print_

In [ ]:
# Step 1: Drop high-missingness amenity columns
cols_to_drop = ["PoolQC", "MiscFeature", "Alley", "Fence"]
df_housing_clean = df_housing.drop(columns=cols_to_drop)

# Step 2: Fill NA in masonry veneer, fireplace, garage, and basement descriptive fields
# Identify columns to fill with 'None' (explicit absence)
masonry_cols = ["MasVnrType"] if "MasVnrType" in df_housing_clean.columns else []
fireplace_cols = ["FireplaceQu"] if "FireplaceQu" in df_housing_clean.columns else []
garage_cols = [
    col
    for col in df_housing_clean.columns
    if col.startswith("Garage") and df_housing_clean[col].dtype == object
]
basement_cols = [
    col
    for col in df_housing_clean.columns
    if col.startswith("Bsmt") and df_housing_clean[col].dtype == object
]

cols_fill_none = masonry_cols + fireplace_cols + garage_cols + basement_cols
for col in cols_fill_none:
    df_housing_clean[col] = df_housing_clean[col].fillna("None")

# Step 3: Remove the row with missing Electrical (should be only one)
df_housing_clean = df_housing_clean[df_housing_clean["Electrical"].notnull()].copy()

# Step 4: Report shape, LotFrontage missing count, and missingness/value summaries for recoded columns
print(f"df_housing_clean shape: {df_housing_clean.shape}")
print(f"LotFrontage missing count: {df_housing_clean['LotFrontage'].isna().sum()}")

print("\nMissing value counts for recoded columns:")
for col in cols_fill_none:
    print(
        f"{col}: {df_housing_clean[col].isna().sum()} missing, unique values: {df_housing_clean[col].unique()}"
    )

# Save cleaned dataframe to workspace
add_to_workspace(df_housing_clean)


### 1.3: Engineer features that capture renovation recency, property age/quality profile, and ROI normalization (price per square foot and total bathrooms).
_Output: print_

In [ ]:
import numpy as np

# Create a feature-engineered copy
df_housing_features = df_housing_clean.copy()

# 1. Age at sale
df_housing_features["age_at_sale"] = (
    df_housing_features["YrSold"] - df_housing_features["YearBuilt"]
)

# 2. Years since remodel
df_housing_features["years_since_remodel"] = (
    df_housing_features["YrSold"] - df_housing_features["YearRemodAdd"]
)

# 3. Recent renovation (within 5 years of sale)
df_housing_features["recent_renovation"] = (
    df_housing_features["years_since_remodel"] <= 5
).astype(int)

# 4. Market timing indicator: pre/post 2008 financial crisis
df_housing_features["post_2008"] = (df_housing_features["YrSold"] >= 2008).astype(int)

# 5. Price per square foot (GrLivArea is above-ground living area)
df_housing_features["price_per_sqft"] = (
    df_housing_features["SalePrice"] / df_housing_features["GrLivArea"]
)

# 6. Total bathrooms (full + 0.5 * half baths, including basement)
df_housing_features["total_bathrooms"] = (
    df_housing_features["FullBath"]
    + 0.5 * df_housing_features["HalfBath"]
    + df_housing_features["BsmtFullBath"]
    + 0.5 * df_housing_features["BsmtHalfBath"]
)

# 7. Old home flag (built before 1950)
df_housing_features["old_home_flag"] = (df_housing_features["YearBuilt"] < 1950).astype(
    int
)

# 8. Low quality flag (OverallQual <= 4)
df_housing_features["low_quality_flag"] = (
    df_housing_features["OverallQual"] <= 4
).astype(int)

# Print summary statistics for new fields
new_fields = [
    "age_at_sale",
    "years_since_remodel",
    "recent_renovation",
    "post_2008",
    "price_per_sqft",
    "total_bathrooms",
    "old_home_flag",
    "low_quality_flag",
]
print("Summary statistics for new engineered features:")
print(df_housing_features[new_fields].describe())

# Print group counts for binary/categorical new fields
print("\nGroup counts for recent_renovation:")
print(df_housing_features["recent_renovation"].value_counts())
print("\nGroup counts for post_2008:")
print(df_housing_features["post_2008"].value_counts())
print("\nGroup counts for old_home_flag:")
print(df_housing_features["old_home_flag"].value_counts())
print("\nGroup counts for low_quality_flag:")
print(df_housing_features["low_quality_flag"].value_counts())

# Save to workspace
add_to_workspace(df_housing_features)


### 1.4: Visualize how renovation recency is distributed over sale years to understand temporal structure and sample balance for subsequent ROI comparisons.
_Output: figure_

In [ ]:
import plotly.express as px

# Prepare data for plotting
df_plot = df_housing_features.copy()
# Map recent_renovation to meaningful labels
recent_renovation_map = {0: "No Recent Renovation", 1: "Recent Renovation"}
df_plot["recent_renovation_label"] = df_plot["recent_renovation"].map(
    recent_renovation_map
)

fig = px.box(
    df_plot,
    x="YrSold",
    y="years_since_remodel",
    color="recent_renovation_label",
    category_orders={"YrSold": sorted(df_plot["YrSold"].unique())},
    labels={
        "YrSold": "Year Sold",
        "years_since_remodel": "Years Since Remodel",
        "recent_renovation_label": "Recent Renovation?",
    },
    title="Renovation Recency by Sale Year: Recent vs. Older Renovations (2006–2010)",
)
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.show()


## Task 2: Identify which neighborhoods exhibit the largest and most consistent price uplift for recently renovated homes and how this uplift varies with property age and quality.

**Acceptance Criteria:** A neighborhood-level summary table quantifies absolute and relative renovation premiums in SalePrice and price_per_sqft for neighborhoods with sufficient sample sizes, highlighting those with both high and stable premiums. Additional summaries show how premiums differ for older vs newer and low- vs higher-quality homes, and how renovation recency buckets influence uplift. At least one visualization clearly ranks neighborhoods by renovation premium.

### 2.1: Compute neighborhood-level renovation premiums comparing recent vs non-recent renovations on both absolute and relative scales, while enforcing minimum group sizes.
_Output: print_

In [ ]:
# Group by Neighborhood and recent_renovation, aggregate metrics
agg_cols = ["SalePrice", "price_per_sqft"]

df_grouped = (
    df_housing_features.groupby(["Neighborhood", "recent_renovation"])
    .agg(
        n_sales=("SalePrice", "count"),
        avg_sale_price=("SalePrice", "mean"),
        avg_price_per_sqft=("price_per_sqft", "mean"),
        std_price_per_sqft=("price_per_sqft", "std"),
    )
    .reset_index()
)

# Pivot to get separate columns for recent/non-recent
pivot_cols = ["n_sales", "avg_sale_price", "avg_price_per_sqft", "std_price_per_sqft"]
df_pivot = df_grouped.pivot(
    index="Neighborhood", columns="recent_renovation", values=pivot_cols
)
# Flatten MultiIndex columns
new_cols = []
for metric, recency in df_pivot.columns:
    label = "recent" if recency == 1 else "not_recent"
    new_cols.append(f"{metric}_{label}")
df_pivot.columns = new_cols

df_pivot = df_pivot.reset_index()

# Only keep neighborhoods with at least 10 sales in each group
mask = (df_pivot["n_sales_recent"] >= 10) & (df_pivot["n_sales_not_recent"] >= 10)
df_neighborhood_premiums = df_pivot[mask].copy()

# Calculate absolute and percentage premiums
# Absolute premium: recent - not_recent
# Percentage premium: (recent - not_recent) / not_recent * 100

df_neighborhood_premiums["sale_price_premium"] = (
    df_neighborhood_premiums["avg_sale_price_recent"]
    - df_neighborhood_premiums["avg_sale_price_not_recent"]
)
df_neighborhood_premiums["sale_price_premium_pct"] = (
    df_neighborhood_premiums["sale_price_premium"]
    / df_neighborhood_premiums["avg_sale_price_not_recent"]
    * 100
)
df_neighborhood_premiums["price_per_sqft_premium"] = (
    df_neighborhood_premiums["avg_price_per_sqft_recent"]
    - df_neighborhood_premiums["avg_price_per_sqft_not_recent"]
)
df_neighborhood_premiums["price_per_sqft_premium_pct"] = (
    df_neighborhood_premiums["price_per_sqft_premium"]
    / df_neighborhood_premiums["avg_price_per_sqft_not_recent"]
    * 100
)
# Stability metric: pooled std of price_per_sqft divided by group mean difference
# Lower is more stable
import numpy as np

n1 = df_neighborhood_premiums["n_sales_recent"]
n0 = df_neighborhood_premiums["n_sales_not_recent"]
s1 = df_neighborhood_premiums["std_price_per_sqft_recent"]
s0 = df_neighborhood_premiums["std_price_per_sqft_not_recent"]
# Pooled std
pooled_std = np.sqrt(((n1 - 1) * s1**2 + (n0 - 1) * s0**2) / (n1 + n0 - 2))
df_neighborhood_premiums["premium_stability"] = pooled_std / df_neighborhood_premiums[
    "price_per_sqft_premium"
].replace(0, np.nan)

# Save to workspace
add_to_workspace(df_neighborhood_premiums)

# Print top neighborhoods by percentage premium per sqft (descending)
df_top = df_neighborhood_premiums.sort_values(
    "price_per_sqft_premium_pct", ascending=False
)[
    [
        "Neighborhood",
        "n_sales_recent",
        "n_sales_not_recent",
        "avg_price_per_sqft_recent",
        "avg_price_per_sqft_not_recent",
        "price_per_sqft_premium",
        "price_per_sqft_premium_pct",
        "premium_stability",
    ]
]
print(
    "Top neighborhoods by % renovation premium per square foot (only neighborhoods with ≥10 sales in each group):"
)
print(df_top.head(10))


### 2.2: Visually rank neighborhoods by renovation premium per square foot while encoding the robustness of each estimate.
_Output: figure_

In [ ]:
import plotly.express as px

# Sort neighborhoods by percentage renovation premium per square foot (descending)
df_plot_bar = df_neighborhood_premiums.sort_values(
    "price_per_sqft_premium_pct", ascending=False
).copy()

fig = px.bar(
    df_plot_bar,
    x="Neighborhood",
    y="price_per_sqft_premium_pct",
    color="n_sales_recent",
    color_continuous_scale="Blues",
    labels={
        "Neighborhood": "Neighborhood",
        "price_per_sqft_premium_pct": "Renovation Premium (% per Sqft)",
        "n_sales_recent": "Recent Renovated Sales",
    },
    title="Neighborhoods Ranked by % Renovation Premium per Sqft (Color: # Recent Renovated Sales)",
)
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.update_layout(
    xaxis={"categoryorder": "array", "categoryarray": df_plot_bar["Neighborhood"]}
)
fig.show()


### 2.3: Determine whether older or lower-quality homes show larger renovation premiums, overall and within promising neighborhoods.
_Output: print_

In [ ]:
# Identify promising neighborhoods: upper quartile of price_per_sqft_premium_pct
import numpy as np

# Compute upper quartile threshold
premium_pct = df_neighborhood_premiums["price_per_sqft_premium_pct"]
q3 = np.percentile(premium_pct, 75)

# Select promising neighborhoods
promising_neighborhoods = df_neighborhood_premiums.loc[
    df_neighborhood_premiums["price_per_sqft_premium_pct"] >= q3, "Neighborhood"
].tolist()

print("Promising neighborhoods (upper quartile of % renovation premium per sqft):")
print(promising_neighborhoods)

# Prepare data for summary tables
# Use df_housing_features for all property-level features
# Add a 'promising_neighborhood' flag

df_housing_features["promising_neighborhood"] = (
    df_housing_features["Neighborhood"].isin(promising_neighborhoods).astype(int)
)


# Helper: function to compute premium summary table
def renovation_premium_summary(df, group_vars):
    # Compute mean price_per_sqft by recent_renovation and group_vars
    df_summary = (
        df.groupby(group_vars + ["recent_renovation"])
        .agg(
            n_sales=("SalePrice", "count"),
            avg_price_per_sqft=("price_per_sqft", "mean"),
        )
        .reset_index()
    )
    # Pivot to get recent/non-recent columns
    df_pivot = df_summary.pivot_table(
        index=group_vars,
        columns="recent_renovation",
        values=["n_sales", "avg_price_per_sqft"],
    )
    # Flatten columns
    df_pivot.columns = [
        f"{col[0]}_{'recent' if col[1]==1 else 'not_recent'}"
        for col in df_pivot.columns
    ]
    df_pivot = df_pivot.reset_index()
    # Calculate premium columns
    df_pivot["price_per_sqft_premium"] = (
        df_pivot["avg_price_per_sqft_recent"]
        - df_pivot["avg_price_per_sqft_not_recent"]
    )
    df_pivot["price_per_sqft_premium_pct"] = (
        df_pivot["price_per_sqft_premium"]
        / df_pivot["avg_price_per_sqft_not_recent"]
        * 100
    )
    return df_pivot


# 1. Overall: Old vs New homes
print("\nRenovation premiums: Old vs New homes (all neighborhoods)")
df_old_new = renovation_premium_summary(df_housing_features, ["old_home_flag"])
print(df_old_new)

# 2. Overall: Low vs High quality homes
print("\nRenovation premiums: Low vs High quality homes (all neighborhoods)")
df_quality = renovation_premium_summary(df_housing_features, ["low_quality_flag"])
print(df_quality)

# 3. Within promising neighborhoods: Old vs New homes
print("\nRenovation premiums: Old vs New homes (promising neighborhoods only)")
df_promising = df_housing_features[
    df_housing_features["promising_neighborhood"] == 1
].copy()
df_old_new_promising = renovation_premium_summary(df_promising, ["old_home_flag"])
print(df_old_new_promising)

# 4. Within promising neighborhoods: Low vs High quality homes
print("\nRenovation premiums: Low vs High quality homes (promising neighborhoods only)")
df_quality_promising = renovation_premium_summary(df_promising, ["low_quality_flag"])
print(df_quality_promising)


### 2.4: Examine how price uplift varies with finer-grained renovation recency buckets beyond a simple recent vs non-recent split.
_Output: print_

In [ ]:
# Create renovation recency buckets from years_since_remodel
# Use df_housing_features, which already has years_since_remodel and promising_neighborhood columns
import numpy as np

# Define buckets: 0-2 years, 3-5 years, 6-10 years, 11+ years
bins = [-np.inf, 2, 5, 10, np.inf]
labels = ["0-2 years", "3-5 years", "6-10 years", "11+ years"]
df_housing_features["renovation_recency_bucket"] = pd.cut(
    df_housing_features["years_since_remodel"], bins=bins, labels=labels, right=True
)

# Overall: count and median price per sqft by bucket
df_bucket_summary = (
    df_housing_features.groupby("renovation_recency_bucket")
    .agg(
        n_sales=("SalePrice", "count"),
        median_price_per_sqft=("price_per_sqft", "median"),
    )
    .reset_index()
)

print("Overall renovation recency buckets: count and median price per sqft")
print(df_bucket_summary)

# Within promising neighborhoods only
df_promising = df_housing_features[
    df_housing_features["promising_neighborhood"] == 1
].copy()
df_bucket_summary_promising = (
    df_promising.groupby("renovation_recency_bucket")
    .agg(
        n_sales=("SalePrice", "count"),
        median_price_per_sqft=("price_per_sqft", "median"),
    )
    .reset_index()
)

print(
    "\nPromising neighborhoods: renovation recency buckets (count and median price per sqft)"
)
print(df_bucket_summary_promising)


### 2.5: Quantify how specific upgrades like kitchen quality, bathroom count, and overall quality relate to price per square foot, both overall and in top renovation neighborhoods.
_Output: print_

In [ ]:
# Define kitchen quality, overall quality, and total bathroom buckets
# Use df_housing_features and df_promising (already filtered for promising neighborhoods)

# 1. Kitchen quality: use as-is (object, ordinal: Ex > Gd > TA > Fa > Po)
# 2. Overall quality bucket: group OverallQual into bins (1-4: Low, 5-6: Average, 7-8: High, 9-10: Luxury)
# 3. Total bathroom bucket: bin total_bathrooms (float) into 1, 1.5-2, 2.5-3, 3.5+

# Prepare buckets on full dataset
df_housing_features = df_housing_features.copy()
df_housing_features["overall_qual_bucket"] = pd.cut(
    df_housing_features["OverallQual"],
    bins=[0, 4, 6, 8, 10],
    labels=["Low (1-4)", "Average (5-6)", "High (7-8)", "Luxury (9-10)"],
    right=True,
)
df_housing_features["total_bath_bucket"] = pd.cut(
    df_housing_features["total_bathrooms"],
    bins=[0, 1.5, 2.5, 3.5, 10],
    labels=["1-1.5", "2-2.5", "3-3.5", "4+"],
    right=False,
)

# Prepare buckets on promising neighborhoods subset
df_promising = df_housing_features[
    df_housing_features["promising_neighborhood"] == 1
].copy()


# Helper function for summary tables
def print_summary_by_bucket(df, bucket_col, bucket_label):
    df_summary = (
        df.groupby(bucket_col)
        .agg(
            median_sale_price=("SalePrice", "median"),
            median_price_per_sqft=("price_per_sqft", "median"),
            n_sales=("SalePrice", "count"),
        )
        .reset_index()
    )
    print(f"\nMedian Sale Price and Price per Sqft by {bucket_label}:")
    print(df_summary)


# --- FULL DATASET ---
print("FULL DATASET:")
print_summary_by_bucket(df_housing_features, "KitchenQual", "Kitchen Quality")
print_summary_by_bucket(
    df_housing_features, "overall_qual_bucket", "Overall Quality Bucket"
)
print_summary_by_bucket(
    df_housing_features, "total_bath_bucket", "Total Bathroom Bucket"
)

# --- PROMISING NEIGHBORHOODS ---
print("\nPROMISING NEIGHBORHOODS (Crawfor, StoneBr, Timber):")
print_summary_by_bucket(df_promising, "KitchenQual", "Kitchen Quality")
print_summary_by_bucket(df_promising, "overall_qual_bucket", "Overall Quality Bucket")
print_summary_by_bucket(df_promising, "total_bath_bucket", "Total Bathroom Bucket")


## Task 3: Use hedonic regression to estimate neighborhood-specific renovation premiums adjusted for property characteristics and market timing, then synthesize all findings to recommend where and how to invest in renovation.

**Acceptance Criteria:** A regression model of log(SalePrice) provides adjusted estimates of the effect of recent renovation, its interactions with neighborhoods and market timing, and the contributions of key upgrades (kitchen quality, bathrooms, overall quality). Diagnostics indicate the model is reasonably well-specified. A ranked, interpretable summary of neighborhoods and property profiles with high, stable, and adjusted renovation premiums is produced, directly answering where renovation investments are most attractive and which upgrade levers matter.

### 3.1: Estimate the adjusted impact of recent renovation on sale price while controlling for size, quality, neighborhood, and market timing, including interactions to capture neighborhood- and period-specific renovation premiums.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 1. Restrict to neighborhoods with sufficient sample size (from df_neighborhood_premiums)
valid_neighborhoods = df_neighborhood_premiums["Neighborhood"].tolist()
df_model = df_housing_features[
    df_housing_features["Neighborhood"].isin(valid_neighborhoods)
].copy()

# 2. Prepare model variables
# Outcome: log(SalePrice)
# Predictors: GrLivArea (size), OverallQual (quality), total_bathrooms, GarageCars, age_at_sale, KitchenQual (ordinal), Neighborhood, post_2008, recent_renovation, old_home_flag
# Interactions: recent_renovation × Neighborhood, recent_renovation × post_2008, recent_renovation × old_home_flag

# Encode KitchenQual as ordered categorical (Ex > Gd > TA > Fa > Po)
kitchen_qual_order = ["Po", "Fa", "TA", "Gd", "Ex"]
df_model["KitchenQual_ord"] = pd.Categorical(
    df_model["KitchenQual"], categories=kitchen_qual_order, ordered=True
).codes

# Drop rows with missing values in model variables
model_vars = [
    "SalePrice",
    "GrLivArea",
    "OverallQual",
    "total_bathrooms",
    "GarageCars",
    "age_at_sale",
    "KitchenQual_ord",
    "Neighborhood",
    "post_2008",
    "recent_renovation",
    "old_home_flag",
]
df_model = df_model.dropna(subset=model_vars).copy()

# 3. Build design matrix
# Main effects
df_X = df_model[
    [
        "GrLivArea",
        "OverallQual",
        "total_bathrooms",
        "GarageCars",
        "age_at_sale",
        "KitchenQual_ord",
        "post_2008",
        "recent_renovation",
        "old_home_flag",
    ]
].copy()
# Neighborhood dummies (drop first for reference)
df_X = pd.concat(
    [
        df_X,
        pd.get_dummies(
            df_model["Neighborhood"], prefix="Neighborhood", drop_first=True
        ),
    ],
    axis=1,
)

# Interactions
# a) recent_renovation × Neighborhood
df_X_int = pd.get_dummies(
    df_model["Neighborhood"], prefix="Neighborhood", drop_first=True
)
for col in df_X_int.columns:
    df_X[f"{col}_x_renov"] = df_X_int[col] * df_model["recent_renovation"]
# b) recent_renovation × post_2008
df_X["recent_renovation_x_post_2008"] = (
    df_model["recent_renovation"] * df_model["post_2008"]
)
# c) recent_renovation × old_home_flag
df_X["recent_renovation_x_old_home"] = (
    df_model["recent_renovation"] * df_model["old_home_flag"]
)

# Add constant
X = sm.add_constant(df_X).astype(float)
y = np.log(df_model["SalePrice"])

# 4. Fit OLS model
model = sm.OLS(y, X).fit()

# 5. Print concise coefficient summary
focus_terms = [
    "recent_renovation",
    "recent_renovation_x_post_2008",
    "recent_renovation_x_old_home",
]
# Add all Neighborhood_x_renov interaction terms
focus_terms += [col for col in X.columns if col.endswith("_x_renov")]
# Also show main upgrade levers
focus_terms += [
    "GrLivArea",
    "OverallQual",
    "total_bathrooms",
    "GarageCars",
    "KitchenQual_ord",
]

print("Key coefficients (log-sale-price model):")
print(model.params[focus_terms].round(3))

# 6. Derive neighborhood-specific implied renovation premiums (as %)
# Reference: baseline is the omitted neighborhood (alphabetically first, due to drop_first=True)
ref_neigh = sorted(valid_neighborhoods)[0]
renov_coef = model.params["recent_renovation"]

neigh_premiums = []
for neigh in valid_neighborhoods:
    # Neighborhood dummy name
    neigh_dummy = f"Neighborhood_{neigh}"
    neigh_x_renov = f"Neighborhood_{neigh}_x_renov"
    # For reference neighborhood, interaction is zero
    if neigh_x_renov in model.params:
        premium_log = renov_coef + model.params[neigh_x_renov]
    else:
        premium_log = renov_coef
    premium_pct = (np.exp(premium_log) - 1) * 100
    neigh_premiums.append(
        {"Neighborhood": neigh, "modeled_renovation_premium_pct": premium_pct}
    )

# 7. Create modeled premium table and save to workspace
df_renovation_premiums_modeled = pd.DataFrame(neigh_premiums)
add_to_workspace(df_renovation_premiums_modeled)

print("\nNeighborhood-specific modeled renovation premiums (% uplift in sale price):")
print(
    df_renovation_premiums_modeled.sort_values(
        "modeled_renovation_premium_pct", ascending=False
    )
)


### 3.2: Assess whether the hedonic model reasonably fits the data and whether assumptions are grossly violated, to ensure that renovation premium estimates are trustworthy.
_Output: figure_

In [ ]:
import plotly.express as px
import numpy as np

# Compute fitted values and residuals from the existing model and modeling sample
y_fitted = model.fittedvalues  # already log(SalePrice)
residuals = model.resid

# Prepare DataFrame for plotting
import pandas as pd

df_residuals = pd.DataFrame({"Fitted Log Sale Price": y_fitted, "Residual": residuals})

fig = px.scatter(
    df_residuals,
    x="Fitted Log Sale Price",
    y="Residual",
    title="Model Residuals vs. Fitted Log Sale Price",
    labels={
        "Fitted Log Sale Price": "Fitted Log Sale Price",
        "Residual": "Residual (log-scale)",
    },
)
# Add horizontal zero-reference line
fig.add_shape(
    type="line",
    x0=df_residuals["Fitted Log Sale Price"].min(),
    x1=df_residuals["Fitted Log Sale Price"].max(),
    y0=0,
    y1=0,
    line=dict(color="red", dash="dash"),
    layer="below",
)
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.show()


### 3.3: Present adjusted renovation premiums by neighborhood from the hedonic model to complement the raw neighborhood premium statistics.
_Output: figure_

In [ ]:
import plotly.express as px

# Prepare data for plotting
# Merge modeled premiums with raw premium table to flag top raw-premium neighborhoods
# The top group is those in the upper quartile of price_per_sqft_premium_pct in df_neighborhood_premiums
import numpy as np

df_plot = df_renovation_premiums_modeled.copy()

# Get the upper quartile threshold for raw premium
q3 = np.percentile(df_neighborhood_premiums["price_per_sqft_premium_pct"], 75)
top_raw_neigh = df_neighborhood_premiums.loc[
    df_neighborhood_premiums["price_per_sqft_premium_pct"] >= q3, "Neighborhood"
].tolist()

# Add a flag for top raw-premium group
raw_premium_map = {
    n: ("Top Raw Premium" if n in top_raw_neigh else "Other")
    for n in df_plot["Neighborhood"]
}
df_plot["raw_premium_group"] = df_plot["Neighborhood"].map(raw_premium_map)

# Sort by modeled premium descending
df_plot = df_plot.sort_values("modeled_renovation_premium_pct", ascending=False)

fig = px.bar(
    df_plot,
    x="Neighborhood",
    y="modeled_renovation_premium_pct",
    color="raw_premium_group",
    color_discrete_map={"Top Raw Premium": "#1f77b4", "Other": "#bbbbbb"},
    labels={
        "Neighborhood": "Neighborhood",
        "modeled_renovation_premium_pct": "Modeled Renovation Premium (%)",
        "raw_premium_group": "Top Raw Premium Group",
    },
    title="Neighborhoods Ranked by Modeled Renovation Premium (%)<br>Blue = Also Top Raw Premium Neighborhood",
)
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.update_layout(
    xaxis={"categoryorder": "array", "categoryarray": df_plot["Neighborhood"]}
)
fig.show()


### 3.4: Explain why certain neighborhoods emerge as top renovation opportunities by comparing their baseline prices, volatility, and upgrade sensitivities to others.
_Output: print_

In [ ]:
# Identify top modeled-premium neighborhoods (from df_renovation_premiums_modeled)
import numpy as np

# Get the upper quartile threshold for modeled renovation premium
premium_modeled = df_renovation_premiums_modeled["modeled_renovation_premium_pct"]
q3_modeled = np.percentile(premium_modeled, 75)
top_modeled_neigh = df_renovation_premiums_modeled.loc[
    df_renovation_premiums_modeled["modeled_renovation_premium_pct"] >= q3_modeled,
    "Neighborhood",
].tolist()

print(
    "Top modeled-premium neighborhoods (upper quartile of modeled renovation premium %):"
)
print(top_modeled_neigh)

# Add a flag to df_housing_features for top-modeled neighborhoods
is_top_modeled = df_housing_features["Neighborhood"].isin(top_modeled_neigh)
df_housing_features["top_modeled_neigh"] = is_top_modeled.astype(int)


# Baseline (non-recently-renovated) price per sqft and std, by group
def baseline_stats(df, group_label):
    df_base = df[df["recent_renovation"] == 0]
    mean_pps = df_base["price_per_sqft"].mean()
    std_pps = df_base["price_per_sqft"].std()
    print(
        f"{group_label}: Baseline (non-recently-renovated) price per sqft: {mean_pps:.2f} (std: {std_pps:.2f})"
    )


print("\nBaseline price per sqft (non-recently-renovated):")
baseline_stats(
    df_housing_features[df_housing_features["top_modeled_neigh"] == 1],
    "Top-modeled neighborhoods",
)
baseline_stats(
    df_housing_features[df_housing_features["top_modeled_neigh"] == 0],
    "Other neighborhoods",
)


# Median price per sqft by kitchen quality and by total bathroom bucket, for both groups
def median_pps_by_bucket(df, bucket_col, bucket_label, group_label):
    df_summary = df.groupby(bucket_col)["price_per_sqft"].median().reset_index()
    print(f"\n{group_label}: Median price per sqft by {bucket_label}:")
    print(df_summary)


# Kitchen quality
median_pps_by_bucket(
    df_housing_features[df_housing_features["top_modeled_neigh"] == 1],
    "KitchenQual",
    "Kitchen Quality",
    "Top-modeled neighborhoods",
)
median_pps_by_bucket(
    df_housing_features[df_housing_features["top_modeled_neigh"] == 0],
    "KitchenQual",
    "Kitchen Quality",
    "Other neighborhoods",
)

# Total bathroom bucket (already exists)
median_pps_by_bucket(
    df_housing_features[df_housing_features["top_modeled_neigh"] == 1],
    "total_bath_bucket",
    "Total Bathroom Bucket",
    "Top-modeled neighborhoods",
)
median_pps_by_bucket(
    df_housing_features[df_housing_features["top_modeled_neigh"] == 0],
    "total_bath_bucket",
    "Total Bathroom Bucket",
    "Other neighborhoods",
)

print(
    "\nInterpretation: The top modeled-premium neighborhoods combine higher baseline prices with strong kitchen and bathroom upgrade premiums, suggesting that buyers in these areas value recent renovations and quality upgrades more than in other neighborhoods. However, higher price variability in some segments may indicate greater investment risk."
)


### 3.5: Cross-check key findings across descriptive and modeled analyses and synthesize a clear answer to where and how renovation investments are most attractive.
_Output: print_

In [ ]:
import numpy as np

# df_compare already exists and contains:
# [Neighborhood, n_sales_not_recent, n_sales_recent, avg_sale_price_not_recent, avg_sale_price_recent, avg_price_per_sqft_not_recent, avg_price_per_sqft_recent, std_price_per_sqft_not_recent, std_price_per_sqft_recent, sale_price_premium, sale_price_premium_pct, price_per_sqft_premium, price_per_sqft_premium_pct, premium_stability, modeled_renovation_premium_pct, raw_rank]

# Compute modeled_rank (was missing in previous attempt)
df_compare = df_compare.copy()
df_compare["modeled_rank"] = df_compare["modeled_renovation_premium_pct"].rank(
    ascending=False, method="min"
)
df_compare["rank_change"] = df_compare["modeled_rank"] - df_compare["raw_rank"]


def classify(row):
    if row["raw_rank"] <= 3 and row["modeled_rank"] <= 3:
        return "Strong in Both"
    elif row["raw_rank"] <= 3 and row["modeled_rank"] > 3:
        return "Weakened After Adjustment"
    elif row["raw_rank"] > 3 and row["modeled_rank"] <= 3:
        return "Strengthened After Adjustment"
    else:
        return "Middle/Lower"


df_compare["premium_alignment"] = df_compare.apply(classify, axis=1)

# Print concise comparison table
cols_to_show = [
    "Neighborhood",
    "n_sales_recent",
    "n_sales_not_recent",
    "price_per_sqft_premium_pct",
    "modeled_renovation_premium_pct",
    "premium_stability",
    "raw_rank",
    "modeled_rank",
    "rank_change",
    "premium_alignment",
]
print("Comparison of Raw vs. Modeled Renovation Premiums by Neighborhood:")
print(df_compare[cols_to_show].sort_values("modeled_rank"))

# Identify neighborhoods in each group
strong_both = df_compare[df_compare["premium_alignment"] == "Strong in Both"][
    "Neighborhood"
].tolist()
weakened = df_compare[df_compare["premium_alignment"] == "Weakened After Adjustment"][
    "Neighborhood"
].tolist()
strengthened = df_compare[
    df_compare["premium_alignment"] == "Strengthened After Adjustment"
]["Neighborhood"].tolist()

print("\nNeighborhoods strong in both raw and modeled premiums:")
print(strong_both)
print("Neighborhoods weakened after adjustment:")
print(weakened)
print("Neighborhoods strengthened after adjustment:")
print(strengthened)

# Property-profile and upgrade findings (from previous feature engineering)
# Use df_housing_features, which already has 'KitchenQual', 'total_bath_bucket', etc.
df_housing_features = df_housing_features.copy()
df_housing_features["top_modeled_neigh"] = (
    df_housing_features["Neighborhood"].isin(strong_both).astype(int)
)

df_top_neigh = df_housing_features[df_housing_features["top_modeled_neigh"] == 1].copy()


def median_pps_by_bucket(df, bucket_col, bucket_label, group_label):
    df_summary = df.groupby(bucket_col)["price_per_sqft"].median().reset_index()
    print(f"\n{group_label}: Median price per sqft by {bucket_label}:")
    print(df_summary)


# Kitchen quality
median_pps_by_bucket(
    df_top_neigh,
    "KitchenQual",
    "Kitchen Quality",
    "Top neighborhoods",
)
# Total bathroom bucket
median_pps_by_bucket(
    df_top_neigh,
    "total_bath_bucket",
    "Total Bathroom Bucket",
    "Top neighborhoods",
)

# Post-2008 interaction: summarize effect from model
post_2008_coef = model.params.get("recent_renovation_x_post_2008", 0.0)
print(f"\nPost-2008 interaction effect (log points): {post_2008_coef:.3f}")
if post_2008_coef > 0.01:
    print("Renovation premium is higher after 2008.")
elif post_2008_coef < -0.01:
    print("Renovation premium is lower after 2008.")
else:
    print("Renovation premium is similar pre- and post-2008.")

# Final ranked recommendation list
print("\n--- Ranked Recommendation List ---")
print(
    "1. Most attractive neighborhoods for renovation investment: "
    + ", ".join(strong_both)
)
print(
    "2. Home types most responsive to renovation: High kitchen quality (Ex, Gd) and homes with 3+ bathrooms show the highest price per sqft."
)
print(
    "3. Upgrades that matter most: Kitchen quality and bathroom count are the strongest levers for price uplift in these neighborhoods."
)
if post_2008_coef > 0.01:
    print(
        "4. The payoff to renovation is even higher after 2008, suggesting a resilient or growing premium in the recent market."
    )
elif post_2008_coef < -0.01:
    print("4. The payoff to renovation is lower after 2008, so timing may matter.")
else:
    print(
        "4. The payoff to renovation is similar before and after 2008, so market timing is less critical."
    )
print(
    "\nNeighborhoods weakened after adjustment may have raw premiums explained by other factors, while those strengthened after adjustment may have hidden upside when controlling for property mix."
)
